In [15]:
# import librairie 
import pandas as pd
import numpy as np
import fastparquet
import seaborn as sns 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder


In [16]:
# préparation du modèle 

df = pd.read_parquet("processed/full_data.parquet", engine="fastparquet")
bakery_orders = df[df['department'] =='bakery']['order_id'].unique()
df_bakery = df[df['order_id'].isin(bakery_orders)].copy()

cart_size = df_bakery.groupby('order_id').agg({
    'product_id': 'count', 
    'order_hour_of_day':'first',
    'order_dow':'first',
    'days_since_prior_order': 'mean'
}).reset_index()

cart_size.rename(columns={'product_id':'cart_size'}, inplace=True)
cart_size = cart_size.fillna(0)

X = cart_size[['order_hour_of_day', 'days_since_prior_order']]
y = cart_size['cart_size']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr = LinearRegression()

lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest MSE:", mse_rf)
print("Random Forest R2:", r2_rf)



# test 
new_order = pd.DataFrame(
    [[10, 7]],
    columns=['order_hour_of_day', 'days_since_prior_order']
)
prediction = rf.predict(new_order) 

Random Forest MSE: 72.98683618501518
Random Forest R2: 0.027042725397962264
